# HA/DR & Replication Tests

**Release:** v1.0.0 | **Date:** 2026-03-17

## Test Cases (Public Preview)
| Capability | Test Case | Target |
|------------|-----------|--------|
| Cross-Region Replication | Create replication group, insert 1M rows | RPO <1hr under 100 TPS |
| Failover Test | Simulate outage, execute failover | RTO <4hr |
| Metadata Validation | Query SYSTEM$GET_ICEBERG_TABLE_INFORMATION | Verify paths, test R/W |

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICEBERG_POC;
USE SCHEMA TESTS;
USE WAREHOUSE COMPUTE_WH;

## Test 1: Cross-Region Replication (Public Preview)
Create replication group and measure lag under load.

In [ ]:
-- Note: Cross-region replication requires secondary account setup
-- This is a template for the replication configuration

-- Step 1: Create replication group (run on PRIMARY account)
-- CREATE REPLICATION GROUP ICEBERG_POC_RG
--     OBJECT_TYPES = DATABASES
--     ALLOWED_DATABASES = ICEBERG_POC
--     ALLOWED_ACCOUNTS = <SECONDARY_ACCOUNT_LOCATOR>
--     REPLICATION_SCHEDULE = '10 MINUTE';

-- Step 2: Create secondary replication group (run on SECONDARY account)
-- CREATE REPLICATION GROUP ICEBERG_POC_RG
--     AS REPLICA OF <PRIMARY_ORG>.<PRIMARY_ACCOUNT>.ICEBERG_POC_RG;

-- Check replication status
SHOW REPLICATION GROUPS;

In [ ]:
-- Create ENCOUNTERS table for replication load testing (1M encounter records)
CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.TESTS.ENCOUNTERS (
    encounter_id            INT,
    patient_id              INT,
    provider_id             INT,
    encounter_date          DATE,
    encounter_type          VARCHAR,
    primary_diagnosis_code  VARCHAR,
    primary_diagnosis_desc  VARCHAR,
    disposition             VARCHAR,
    total_charge            DECIMAL(10,2)
)
CATALOG = SNOWFLAKE
EXTERNAL_VOLUME = EXVOL
ICEBERG_VERSION = 3;

-- Insert 1M encounter records for replication load test
INSERT INTO ICEBERG_POC.TESTS.ENCOUNTERS
SELECT
    SEQ4() + 3001                                                                                          AS encounter_id,
    SEQ4() % 100000 + 1001                                                                                AS patient_id,
    SEQ4() % 1000   + 2001                                                                                AS provider_id,
    DATEADD(day, -(SEQ4() % 730), CURRENT_DATE())                                                        AS encounter_date,
    ARRAY_CONSTRUCT('Outpatient Visit','Emergency','Inpatient','Specialist Referral','Preventive Care')[SEQ4() % 5]::VARCHAR AS encounter_type,
    ARRAY_CONSTRUCT('E11.9','I10','J06.9','M54.5','Z00.00','F32.1','K21.0','N39.0')[SEQ4() % 8]::VARCHAR  AS primary_diagnosis_code,
    ARRAY_CONSTRUCT('Type 2 diabetes','Hypertension','Upper respiratory infection','Low back pain','Annual wellness visit','Depression','GERD','UTI')[SEQ4() % 8]::VARCHAR AS primary_diagnosis_desc,
    ARRAY_CONSTRUCT('Discharged','Admitted','Transferred','Observation')[SEQ4() % 4]::VARCHAR              AS disposition,
    ROUND((SEQ4() % 9900 + 100) / 10.0, 2)                                                               AS total_charge
FROM TABLE(GENERATOR(ROWCOUNT => 1000000));

SELECT COUNT(*) AS row_count FROM ICEBERG_POC.TESTS.ENCOUNTERS;

## Test 2: Metadata Validation
Query SYSTEM$GET_ICEBERG_TABLE_INFORMATION and verify metadata paths.

In [ ]:
-- Get Iceberg table metadata for ENCOUNTERS (validate replication readiness)
SELECT SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.TESTS.ENCOUNTERS') AS iceberg_info;

-- Parse and display metadata details
SELECT
    PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.TESTS.ENCOUNTERS')):metadataLocation::STRING AS metadata_location,
    PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.TESTS.ENCOUNTERS')):status::STRING           AS status;

-- Show all Iceberg tables and their metadata
SHOW ICEBERG TABLES IN SCHEMA ICEBERG_POC.TESTS;

## Test 3: Failover Simulation
Document failover procedure and measure RTO/RPO targets (RTO < 4hr, RPO < 1hr).

In [ ]:
-- Failover Procedure Template (run on SECONDARY account during DR event)
--
-- Step 1: Check replication lag before failover
-- SELECT REPLICATION_GROUP_NAME, LAST_REFRESH_TIME, NEXT_SCHEDULED_REFRESH_TIME
-- FROM TABLE(INFORMATION_SCHEMA.REPLICATION_GROUP_REFRESH_HISTORY())
-- ORDER BY LAST_REFRESH_TIME DESC LIMIT 5;
--
-- Step 2: Execute failover (promotes secondary to primary)
-- ALTER REPLICATION GROUP ICEBERG_POC_RG FAILOVER;
--
-- Step 3: Verify ENCOUNTERS table accessibility
-- SELECT COUNT(*) FROM ICEBERG_POC.TESTS.ENCOUNTERS;
--
-- Step 4: Test R/W operations on healthcare data
-- INSERT INTO ICEBERG_POC.TESTS.ENCOUNTERS VALUES
--     (9999999, 1001, 2001, CURRENT_DATE(), 'Emergency', 'I10', 'Hypertension', 'Admitted', 850.00);
-- SELECT * FROM ICEBERG_POC.TESTS.ENCOUNTERS WHERE encounter_id = 9999999;

-- Record failover metrics
-- RTO Target: < 4 hours
-- RPO Target: < 1 hour (encounter records replicated within 10-minute schedule)

SELECT 'Failover procedure documented - requires secondary account for testing' AS status;